# Project 1: Annual Median AQI Across U.S. Counties

This project asks: **How are annual median AQI values distributed across monitored U.S. counties in 2022?**

The data come from the EPA's [Annual AQI by County, 2022](https://aqs.epa.gov/aqsweb/airdata/annual_aqi_by_county_2022.zip) file, listed on the [AirData download page](https://aqs.epa.gov/aqsweb/airdata/download_files.html). Each row represents one county with AQI monitoring data in 2022. **Median AQI** is the median daily AQI for that county's monitored days. Lower AQI values generally indicate better air quality. Counties without monitoring data are not included.

## Pandas

In [1]:
from pathlib import Path

import pandas as pd

data_path = Path("data/annual_aqi_by_county_2022.csv")
if not data_path.exists():
    data_path = Path("project_1/data/annual_aqi_by_county_2022.csv")

df = pd.read_csv(data_path)
print(f"Rows: {len(df):,}")
df.head()

Rows: 1,004


,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10
0,Alabama,Baldwin,2022,281,237,44,0,0,0,0,97,52,39,0,0,201,80,0
1,Alabama,Clay,2022,115,91,24,0,0,0,0,68,56,34,0,0,0,115,0
2,Alabama,DeKalb,2022,364,323,41,0,0,0,0,97,52,38,0,0,305,59,0
3,Alabama,Elmore,2022,240,232,8,0,0,0,0,74,44,33,0,0,240,0,0
4,Alabama,Etowah,2022,359,224,135,0,0,0,0,93,60,46,0,0,93,266,0


In [2]:
median_aqi = pd.to_numeric(df["Median AQI"], errors="coerce")
clean_aqi = median_aqi.dropna()

pandas_mean = clean_aqi.mean()
pandas_median = clean_aqi.median()
pandas_modes = sorted(clean_aqi.mode().tolist())

print(f"Mean: {pandas_mean:.2f}")
print(f"Median: {pandas_median:g}")
print("Mode(s):", [f"{float(value):g}" for value in pandas_modes])

Mean: 39.34
Median: 40
Mode(s): ['44']


## The hard way: Python standard library

This section reads the CSV again and does not use the pandas DataFrame.

In [3]:
import csv
from pathlib import Path

csv_path = Path("data/annual_aqi_by_county_2022.csv")
if not csv_path.exists():
    csv_path = Path("project_1/data/annual_aqi_by_county_2022.csv")

values = []
with csv_path.open(newline="", encoding="utf-8-sig") as csv_file:
    for row in csv.DictReader(csv_file):
        try:
            values.append(float(row["Median AQI"]))
        except (TypeError, ValueError):
            continue

# Mean
python_mean = sum(values) / len(values)

# Median
sorted_values = sorted(values)
n = len(sorted_values)
middle = n // 2
if n % 2 == 1:
    python_median = sorted_values[middle]
else:
    python_median = (sorted_values[middle - 1] + sorted_values[middle]) / 2

# Mode
counts = {}
for value in values:
    counts[value] = counts.get(value, 0) + 1

highest_count = 0
for count in counts.values():
    if count > highest_count:
        highest_count = count

python_modes = []
for value, count in counts.items():
    if count == highest_count:
        python_modes.append(value)
python_modes = sorted(python_modes)

print(f"Mean: {python_mean:.2f}")
print(f"Median: {python_median:g}")
print("Mode(s):", [f"{value:g}" for value in python_modes])

Mean: 39.34
Median: 40
Mode(s): ['44']


## Distribution

In [4]:
bin_width = 10
first_bin = int(min(values) // bin_width * bin_width)
last_bin = int(max(values) // bin_width * bin_width)
bins = {start: 0 for start in range(first_bin, last_bin + bin_width, bin_width)}

for value in values:
    bin_start = int(value // bin_width * bin_width)
    bins[bin_start] += 1

max_bar_length = 40
scale = max(bins.values()) / max_bar_length
if scale < 1:
    scale = 1

print("Distribution of 2022 Median AQI Across Monitored U.S. Counties")
print("Median AQI range | Monitored counties")
for start, count in bins.items():
    bar_length = 0 if count == 0 else max(1, round(count / scale))
    bar = "█" * bar_length
    print(f"{start:>2}–{start + 9:<2} | {bar:<40} {count:>4,}")

Distribution of 2022 Median AQI Across Monitored U.S. Counties
Median AQI range | Monitored counties
 0–9  | █                                          14
10–19 | ████                                       46
20–29 | ██████                                     62
30–39 | ██████████████████████████████            334
40–49 | ████████████████████████████████████████  448
50–59 | ████████                                   84
60–69 | █                                           9
70–79 | █                                           6
80–89 | █                                           1


## Findings

Across the 1,004 monitored counties, the mean annual median AQI was 39.34, the median was 40, and the mode was 44. The mean and median are very close, so they give a similar picture of the center of the distribution. The mode means that an annual median AQI of 44 appeared more often than any other value.

Most counties were concentrated between 30 and 49, with 40–49 as the largest 10-point range. Only a small number of counties had annual median AQI values of 60 or higher. The lower-value side is longer, which is consistent with the mean being slightly below the median.

One caveat is that this file only includes counties with AQI monitoring data. Counties may also have different numbers of monitored days, and each county is weighted equally regardless of population. A county's median describes a typical monitored day, not its worst pollution event or every resident's personal exposure.